In [1]:
import numpy as np
import glob
import os
import joblib
import tensorflow as tf
from pyscf import gto, scf
from tensorflow.keras import layers, models, callbacks, optimizers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# =============================================================================
# 0. CONFIGURATION & DIRECTORY SETUP
# =============================================================================
N_ATOMS = 80 
SYSTEM_NAME = f"H{N_ATOMS}"
CHECKPOINT_PATTERN = f"data_checkpoint_h{N_ATOMS}_run*_rank*_step*.npz" 

DEPLOY_DIR = "deployment_objects"
os.makedirs(DEPLOY_DIR, exist_ok=True)
MODEL_SAVE_NAME  = os.path.join(DEPLOY_DIR, f"NN_{SYSTEM_NAME}_DeltaHF.keras")

print(f">>> STARTING DELTA-LEARNING PIPELINE FOR: {SYSTEM_NAME}")

# =============================================================================
# 1. PHYSICS SETUP & REFERENCE GENERATION
# =============================================================================
print(f"\n>>> 1. Generating Physics Constants & Baseline (HF)...")

mol = gto.M(
    atom=[("H", 0.74 * j, 0, 0) for j in range(N_ATOMS)], 
    basis="sto-6g", verbose=0, spin=0
)
mf = scf.UHF(mol)
mf.kernel()

S = mf.get_ovlp()
eigvals, eigvecs = np.linalg.eigh(S)
S_sqrt = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T
S_inv_sqrt = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T

h_core_ao = mf.get_hcore()
h_core_lowdin = S_inv_sqrt @ h_core_ao @ S_inv_sqrt

# Exact HF Baseline
P_hf_spin = mf.make_rdm1()
P_hf_ao = P_hf_spin[0] + P_hf_spin[1] if P_hf_spin.ndim == 3 else P_hf_spin
# Contravariant transformation for Density Matrix
P_hf_lowdin = S_sqrt @ P_hf_ao @ S_sqrt
E_hf = mf.e_tot

C_sim = mf.mo_coeff[0] 

np.save(os.path.join(DEPLOY_DIR, "S_sqrt.npy"), S_sqrt)
np.save(os.path.join(DEPLOY_DIR, "C_sim.npy"), C_sim)
np.save(os.path.join(DEPLOY_DIR, "h_core_lowdin.npy"), h_core_lowdin)
np.save(os.path.join(DEPLOY_DIR, "P_hf_lowdin.npy"), P_hf_lowdin)
np.save(os.path.join(DEPLOY_DIR, "E_hf.npy"), np.array([E_hf]))

# =============================================================================
# 2. BULLETPROOF CUSTOM LOSS
# =============================================================================
@tf.keras.utils.register_keras_serializable()
def log_cosh_loss(y_true, y_pred):
    """Robust loss with strict shape enforcement to prevent broadcasting bugs."""
    y_true = tf.cast(tf.reshape(y_true, tf.shape(y_pred)), dtype=y_pred.dtype)
    return tf.reduce_mean(tf.math.log(tf.math.cosh(y_pred - y_true)))

# =============================================================================
# 3. LOAD & PROCESS WALKER DATA (ACCELERATED)
# =============================================================================
print(f"\n>>> 3. Loading Walker Checkpoints in Parallel...")
files = sorted(glob.glob(CHECKPOINT_PATTERN))
if not files: raise ValueError("No files found!")

nbasis = h_core_ao.shape[0]

# Helper function for parallel loading
def load_checkpoint(f):
    with np.load(f) as data:
        # Extracting immediately to close the file handler
        return data['GA'], data['GB'], data['E']

# 1. Parallelize I/O using all available CPU cores
# This bypasses the sequential wait time of reading .npz files
results = joblib.Parallel(n_jobs=-1)(joblib.delayed(load_checkpoint)(f) for f in files)

# Unpack results into continuous arrays
GA_raw = np.concatenate([r[0] for r in results], axis=0).reshape(-1, nbasis, nbasis)
GB_raw = np.concatenate([r[1] for r in results], axis=0).reshape(-1, nbasis, nbasis)
E_raw  = np.concatenate([r[2] for r in results], axis=0).real.reshape(-1)

# Free up memory before heavy matrix math
del results

print("    Transforming Walkers AO -> Lowdin (BLAS Accelerated)...")
# 2. Swap einsum for batched matrix multiplication (@)
# NumPy broadcasts the `@` operator over the batch dimension using multi-threaded BLAS
GA_lowdin = S_sqrt @ GA_raw @ S_sqrt
GB_lowdin = S_sqrt @ GB_raw @ S_sqrt

# =============================================================================
# 4. PREPARE TRAINING DATA (FIXED SHAPES)
# =============================================================================
print("\n>>> 4. Preparing Physics-Informed Delta Features...")

delta_E_raw = E_raw - E_hf
P_total = GA_lowdin + GB_lowdin
delta_P = P_total - P_hf_lowdin

# Exact 1-Body Delta Energy
E_1B_delta = np.einsum('ij, bji -> b', h_core_lowdin, delta_P).real

# Pure Correlation Residual
y_corr_raw = delta_E_raw - E_1B_delta

median_e = np.median(y_corr_raw)
mad_e = np.median(np.abs(y_corr_raw - median_e))
mask = (y_corr_raw > (median_e - 4 * mad_e)) & (y_corr_raw < (median_e + 4 * mad_e))

delta_P_clean = delta_P[mask]
y_corr_clean  = y_corr_raw[mask]

X_final = np.stack([np.real(delta_P_clean), np.imag(delta_P_clean)], axis=-1)
X_flipped = np.flip(X_final, axis=(1, 2))
y_flipped = y_corr_clean

X_augmented = np.concatenate([X_final, X_flipped], axis=0)
y_augmented = np.concatenate([y_corr_clean, y_flipped], axis=0)

X_train, X_test, y_train, y_test = train_test_split(X_augmented, y_augmented, test_size=0.2, random_state=42)

# STANDARDIZATION (Keeping targets strictly 2D for Keras)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)

X_scaler = StandardScaler()
X_train_scaled = X_scaler.fit_transform(X_train_flat).reshape(X_train.shape)
X_test_scaled  = X_scaler.transform(X_test_flat).reshape(X_test.shape)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)) # No flatten()
y_test_scaled  = y_scaler.transform(y_test.reshape(-1, 1))      # No flatten()

joblib.dump(X_scaler, os.path.join(DEPLOY_DIR, "X_scaler.save"))
joblib.dump(y_scaler, os.path.join(DEPLOY_DIR, "y_scaler.save"))

# =============================================================================
# 5. BUILD MLP BOTTLENECK MODEL
# =============================================================================
print("\n>>> 5. Building Regularized Correlation Model...")

def create_correlation_model(input_shape):
    p_in = layers.Input(shape=input_shape)
    x = layers.Flatten()(p_in)
    
    reg = regularizers.l2(1e-4)
    x = layers.Dense(256, activation='swish', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(128, activation='swish', kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Dense(64, activation='swish', kernel_regularizer=reg)(x)
    
    e_corr_scaled = layers.Dense(1, name="Scaled_Corr_Prediction")(x)
    return models.Model(inputs=p_in, outputs=e_corr_scaled)

model = create_correlation_model(input_shape=(X_train.shape[1], X_train.shape[2], 2))
model.compile(loss=log_cosh_loss, optimizer=optimizers.Adamax(learning_rate=1e-3), metrics=['mae'])

history = model.fit(
    X_train_scaled, y_train_scaled,
    validation_data=(X_test_scaled, y_test_scaled),
    epochs=1000, batch_size=128, verbose=1,
    callbacks=[
        callbacks.EarlyStopping(patience=40, restore_best_weights=True, monitor='val_loss'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=15, verbose=1)
    ]
)

# =============================================================================
# 6. EVALUATION
# =============================================================================
print("\n>>> 6. Final Evaluation...")
preds_scaled = model.predict(X_test_scaled)

preds_corr = y_scaler.inverse_transform(preds_scaled).flatten()
y_test_corr = y_scaler.inverse_transform(y_test_scaled).flatten()

mae = np.mean(np.abs(preds_corr - y_test_corr)) * 1000 
print(f"    Test Correlation MAE: {mae:.4f} mHa")

model.save(MODEL_SAVE_NAME)
print(f"    Model saved to {MODEL_SAVE_NAME}")

2026-03-01 14:40:58.350593: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


>>> STARTING DELTA-LEARNING PIPELINE FOR: H80

>>> 1. Generating Physics Constants & Baseline (HF)...

>>> 3. Loading Walker Checkpoints in Parallel...
    Transforming Walkers AO -> Lowdin (BLAS Accelerated)...

>>> 4. Preparing Physics-Informed Delta Features...

>>> 5. Building Regularized Correlation Model...


I0000 00:00:1772394113.673794 1908910 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 39244 MB memory:  -> device: 0, name: NVIDIA A40, pci bus id: 0000:23:00.0, compute capability: 8.6
2026-03-01 14:42:04.443312: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 1922457600 exceeds 10% of free system memory.
2026-03-01 14:42:06.878747: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 1922457600 exceeds 10% of free system memory.


Epoch 1/1000


2026-03-01 14:42:09.515068: I external/local_xla/xla/service/service.cc:163] XLA service 0x7fd2880091c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-01 14:42:09.515110: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA A40, Compute Capability 8.6
2026-03-01 14:42:09.577396: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-01 14:42:09.787598: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2026-03-01 14:42:09.956317: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-01 14:42:09.956370: I external/local

...

290/294 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.3551 - mae: 0.6435

2026-03-01 14:42:28.212715: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-01 14:42:28.212767: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-01 14:42:28.212779: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-01 14:42:28.212793: I external/l

294/294 ━━━━━━━━━━━━━━━━━━━━ 41s 75ms/step - loss: 0.3538 - mae: 0.6416 - val_loss: 0.1705 - val_mae: 0.3467 - learning_rate: 0.0010
Epoch 2/1000
294/294 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2004 - mae: 0.4065 - val_loss: 0.1429 - val_mae: 0.2855 - learning_rate: 0.0010
Epoch 3/1000
294/294 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1695 - mae: 0.3489 - val_loss: 0.1331 - val_mae: 0.2681 - learning_rate: 0.0010
Epoch 4/1000
294/294 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1549 - mae: 0.3210 - val_loss: 0.1231 - val_mae: 0.2481 - learning_rate: 0.0010
Epoch 5/1000
294/294 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1421 - mae: 0.2988 - val_loss: 0.1164 - val_mae: 0.2362 - learning_rate: 0.0010
Epoch 6/1000
294/294 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1319 - mae: 0.2819 - val_loss: 0.1115 - val_mae: 0.2328 - learning_rate: 0.0010
Epoch 7/1000
294/294 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1250 - mae: 0.2741 - val_loss: 0.1057 - val_mae: 0.2279 - learning_rate: 0.0010
Epo

2026-03-01 14:53:22.719828: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-01 14:53:22.719879: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-01 14:53:24.701636: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_47', 16 bytes spill stores, 16 bytes spill loads

2026-03-01 14:53:27.239108: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : R

294/294 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step
    Test Correlation MAE: 33.4300 mHa
    Model saved to deployment_objects/NN_H80_DeltaHF.keras
